In [1]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
from data_io import DataIO #Custom IO file
run_label = '2023-11-06' ## change this as needed
path = "/gpfs/gibbs/project/david_moore/aj487/Data_WL110/Outgassing_Setup/20231106/OutgassingData_{}.h5".format(run_label, run_label)
IO = DataIO(path)

In [2]:
print(path)

/gpfs/gibbs/project/david_moore/aj487/Data_WL110/Outgassing_Setup/20231106/OutgassingData_2023-11-06.h5


In [3]:
IO.GetTimeData()
timedataframe = IO.timedata
print(timedataframe)

0   2023-11-06 16:28:43
0   2023-11-06 16:29:17
0   2023-11-06 16:29:52
0   2023-11-06 16:30:26
0   2023-11-06 16:31:01
            ...        
0   2023-11-08 16:26:35
0   2023-11-08 16:27:05
0   2023-11-08 16:27:35
0   2023-11-08 16:28:06
0   2023-11-08 16:28:37
Length: 5648, dtype: datetime64[ns]


In [4]:
IO.GetTECData()
tecdataframe = IO.tecdata
print(tecdataframe)

    Set Temp
0        0.0
0        0.0
0        0.0
0        0.0
0        0.0
..       ...
0       50.0
0       50.0
0       50.0
0       50.0
0       50.0

[5648 rows x 1 columns]


In [5]:
IO.GetPressureData()
pressuredataframe = IO.pressuredata
print(pressuredataframe)

    Total Pressure
0         0.000082
0         0.000082
0         0.000082
0         0.000082
0         0.000081
..             ...
0         0.000016
0         0.000017
0         0.000017
0         0.000017
0         0.000016

[5648 rows x 1 columns]


In [6]:
IO.GetOmegaData()
omegadataframe = IO.omegadata
print(omegadataframe)

     CH1   CH2
0   24.7  24.7
0   24.8  24.6
0   24.8  24.7
0   24.8  24.6
0   24.8  24.6
..   ...   ...
0   46.2  45.0
0   46.3  45.0
0   46.3  45.0
0   46.3  45.0
0   46.3  45.0

[5648 rows x 2 columns]


In [7]:
IO.GetRGAData()
rgadataframe = IO.rgadata
rgadataframe[rgadataframe < 0] = 1e-15 #negative pressure values recorded due to noise fluctutation below the RGA's sensitivity
print(rgadataframe['32.00amu'])

data run = 1       1.000000e-15
data run = 2       1.000000e-15
data run = 3       1.000000e-15
data run = 4       1.000000e-15
data run = 5       3.300000e-12
                       ...     
data run = 5644    1.084300e-09
data run = 5645    1.052200e-09
data run = 5646    1.077650e-09
data run = 5647    1.110000e-09
data run = 5648    1.132600e-09
Name: 32.00amu, Length: 5648, dtype: float64


# Data manipulation

In [8]:
start_datetime = timedataframe.iloc[0]
gases = ['H2', 'H2O', 'N2', 'O2', 'CO2']
gas_masses = ['2.00amu', '18.00amu', '28.00amu', '32.00amu', '44.00amu']

In [9]:
final_tables = []
for idx, gas_mass in enumerate(gas_masses):
    time_column = (timedataframe - start_datetime) / np.timedelta64(1, 's')
    rga_column = rgadataframe[gas_masses[idx]].reset_index(drop=True)  # Reset index of rga_column to prevent NaN
    pressure_column = pressuredataframe
    temp1_column = omegadataframe['CH1']
    temp2_column = omegadataframe['CH2']
    tec_column = tecdataframe
    
    # Collecting columns in a dataframe
    gas_final_data = pd.DataFrame(data=time_column)
    gas_final_data.columns = ['Exposure_time']
    
    # Merge rga_column with gas_final_data and fill NaN values
    gas_final_data['Partial_pressure'] = rga_column.values #the attribute .values prevents the index resetting from duplicate values
    
    gas_final_data['Total_pressure'] = pressure_column
    
    # Celsius to Kelvin
    gas_final_data['CH1_temp'] = temp1_column + 273.2
    gas_final_data['CH2_temp'] = temp2_column + 273.2
    gas_final_data['Mean_temp'] = (gas_final_data['CH1_temp'] + gas_final_data['CH2_temp']) / 2.0
    gas_final_data['TEC_temp'] = tec_column
    
    final_tables.append(gas_final_data)
    print(gas_final_data)

    Exposure_time  Partial_pressure  Total_pressure  CH1_temp  CH2_temp  \
0             0.0      1.000000e-15        0.000082     297.9     297.9   
0            34.0      1.860000e-11        0.000082     298.0     297.8   
0            69.0      1.000000e-15        0.000082     298.0     297.9   
0           103.0      1.000000e-15        0.000082     298.0     297.8   
0           138.0      1.300000e-12        0.000081     298.0     297.8   
..            ...               ...             ...       ...       ...   
0        172672.0      3.292000e-09        0.000016     319.4     318.2   
0        172702.0      2.994050e-09        0.000017     319.5     318.2   
0        172732.0      3.307350e-09        0.000017     319.5     318.2   
0        172763.0      3.332600e-09        0.000017     319.5     318.2   
0        172794.0      3.318600e-09        0.000016     319.5     318.2   

    Mean_temp  TEC_temp  
0      297.90       0.0  
0      297.90       0.0  
0      297.95       0

In [10]:
new_path = "/gpfs/gibbs/project/david_moore/aj487/Data_WL110/Outgassing_Setup/20231106"
hdf_name = '{}/{}.h5'.format(new_path, run_label)
for idx, gas in enumerate(gases):
    final_tables[idx].sort_values(by='Exposure_time', inplace=True) # one more sort just to be sure
    final_tables[idx].to_hdf(hdf_name, key=gas)
print(final_tables)

[    Exposure_time  Partial_pressure  Total_pressure  CH1_temp  CH2_temp  \
0             0.0      1.000000e-15        0.000082     297.9     297.9   
0            34.0      1.860000e-11        0.000082     298.0     297.8   
0            69.0      1.000000e-15        0.000082     298.0     297.9   
0           103.0      1.000000e-15        0.000082     298.0     297.8   
0           138.0      1.300000e-12        0.000081     298.0     297.8   
..            ...               ...             ...       ...       ...   
0        172672.0      3.292000e-09        0.000016     319.4     318.2   
0        172702.0      2.994050e-09        0.000017     319.5     318.2   
0        172732.0      3.307350e-09        0.000017     319.5     318.2   
0        172763.0      3.332600e-09        0.000017     319.5     318.2   
0        172794.0      3.318600e-09        0.000016     319.5     318.2   

    Mean_temp  TEC_temp  
0      297.90       0.0  
0      297.90       0.0  
0      297.95       